# Systematic Scaling Study — Colab Runner

**Tinker RL Project — PES University MTech Capstone (Group 6)**

Runs all three blocks of the systematic scaling study using Unsloth + TRL GRPOTrainer
as a drop-in replacement for Atropos + Tinker RL.

| Block | Purpose | Models | GPU |
|-------|---------|--------|-----|
| A | Instruction-tuning isolation | Llama-3.1-8B base vs instruct | A100 40GB |
| B | Family isolation | Qwen3-8B base vs Llama-3.1-8B base | A100 40GB |
| C | Size ladder | Qwen3: 0.6B → 1.7B → 4B → 8B → 14B → 30B-MoE | T4 (tiny) / A100 (large) |

**Recommended runtimes:**
- T4 (free): Block C models ≤ 1.7B only
- A100 40GB (Colab Pro): Block A, B, Block C ≤ 8B
- A100 80GB (Vast.ai): Block C 14B / 30B-MoE

---

> Set `BLOCK` and `MODEL_CONFIG` in **Cell 3** before running.

## Step 1 — Install Dependencies

Run once per Colab session. Takes ~4 minutes on a fresh runtime.

In [ ]:
# Check GPU
!nvidia-smi | head -20

In [ ]:
# Install Unsloth (Colab-optimised build)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# TRL ≥ 0.9 for GRPOTrainer
!pip install -q 'trl>=0.9.0' 'peft>=0.12.0' 'accelerate>=0.33.0'

# Reward verification libraries
!pip install -q math-verify latex2sympy2-extended

# Other deps
!pip install -q datasets wandb pyyaml

print('\n✓ Dependencies installed')

## Step 2 — Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/pes-llm-research/tinker-rl-lab.git'  # update if needed
REPO_DIR = '/content/tinker-rl-lab'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}/atropos
print(f'Working directory: {os.getcwd()}')

## Step 3 — Credentials & Run Config

**Edit this cell** to select which experiment to run.

> WandB is **required** — all metrics stream there in real time. Get your API key at https://wandb.ai/authorize

In [ ]:
import os

# ── Credentials (REQUIRED) ──────────────────────────────────────────────────
HF_TOKEN      = 'hf_YOUR_TOKEN_HERE'       # HuggingFace token (required for gated models)
WANDB_API_KEY = 'YOUR_WANDB_API_KEY_HERE'  # WandB API key — get at wandb.ai/authorize

assert WANDB_API_KEY != 'YOUR_WANDB_API_KEY_HERE', \
    "Set your WandB API key above before running."

os.environ['HF_TOKEN']      = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_SILENT']  = 'false'   # show WandB output in cell

# ── Experiment selector ──────────────────────────────────────────────────────
# Choose ONE config per run.
#
# Block A — instruction-tuning isolation (run BOTH for comparison)
#   'configs/gsm8k_llama_8b_base.yaml'      Llama-3.1-8B BASE  ← new
#   'configs/gsm8k_llama_8b.yaml'           Llama-3.1-8B Instruct (already run)
#
# Block B — family isolation (Qwen3-8B already run; run the base Llama for this block)
#   'configs/gsm8k_llama_8b_base.yaml'
#   'configs/gsm8k_qwen_8b.yaml'            (already run)
#
# Block C — size ladder (smallest first)
#   'configs/gsm8k_qwen_0_6b.yaml'          0.6B  — T4 OK
#   'configs/gsm8k_qwen_1_7b.yaml'          1.7B  — T4 OK
#   'configs/gsm8k_qwen_4b.yaml'            4B    — T4 (4-bit) or A100
#   'configs/gsm8k_qwen_8b.yaml'            8B    — A100 (already run)
#   'configs/gsm8k_qwen_14b.yaml'           14B   — A100 40-80 GB
#   'configs/gsm8k_qwen_30b_moe.yaml'       30B   — A100 80 GB

MODEL_CONFIG = 'configs/gsm8k_qwen_0_6b.yaml'   # ← CHANGE THIS
SEED         = 42

print(f'Config  : {MODEL_CONFIG}')
print(f'Seed    : {SEED}')
print(f'WandB   : enabled (project=tinker-rl-scaling, group=systematic-scaling-study)')

In [ ]:
# Verify WandB login before starting a potentially multi-hour run
import wandb
wandb.login(key=WANDB_API_KEY)
print('WandB login OK')

## Step 4 — Verify Config

In [ ]:
import yaml, json

with open(MODEL_CONFIG) as f:
    cfg_raw = yaml.safe_load(f)

env = cfg_raw.get('env', {})
t   = cfg_raw.get('tinker', {})
oi  = (cfg_raw.get('openai') or [{}])[0]

model_name = oi.get('model_name') or env.get('tokenizer_name')
params_hint = model_name.split('/')[-1] if model_name else 'unknown'

print(f'Model        : {model_name}')
print(f'Steps        : {env.get("total_steps", 50)}')
print(f'Batch / group: {env.get("batch_size", 128)} / {env.get("group_size", 16)}')
print(f'LoRA rank    : {t.get("lora_rank", 32)}')
print(f'Checkpoint   : {t.get("checkpoint_dir", "./checkpoints/run/")}')
print(f'WandB run    : {t.get("wandb_run_name", "grpo-run")}')

## Step 5 — Run Training

Expected times (50 steps, batch=128/group=16 → 8 groups × 16 completions):
- 0.6B on T4:  ~25 min
- 1.7B on T4:  ~55 min
- 8B on A100:  ~35 min
- 14B on A100 80GB: ~70 min

In [ ]:
# Run training — output streams in real time
!python train_grpo_unsloth.py \
    --config {MODEL_CONFIG} \
    --seed {SEED} \
    --wandb_key {WANDB_API_KEY}

## Step 6 — Read Results

The trainer saves a CSV at `<checkpoint_dir>/reward_log.csv`.

In [ ]:
import yaml, os, csv
import matplotlib.pyplot as plt

with open(MODEL_CONFIG) as f:
    cfg_raw = yaml.safe_load(f)

t = cfg_raw.get('tinker', {})
oi = (cfg_raw.get('openai') or [{}])[0]
env = cfg_raw.get('env', {})

checkpoint_dir = t.get('checkpoint_dir', './checkpoints/run/')
csv_path = os.path.join(checkpoint_dir, 'reward_log.csv')
model_name = oi.get('model_name') or env.get('tokenizer_name', 'unknown')

steps, rewards = [], []
with open(csv_path) as f:
    for row in csv.DictReader(f):
        steps.append(int(row['step']))
        rewards.append(float(row['mean_reward']))

print(f'Model : {model_name}')
print(f'Steps : {len(steps)}')
print(f'Step-0 accuracy : {rewards[0]:.4f}')
print(f'Final accuracy  : {rewards[-1]:.4f}')
print(f'\nFull reward list (for NEW_RUNS dict):')
print(f'rewards = {rewards}')

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, linewidth=2, color='#2196F3')
plt.xlabel('Training Step')
plt.ylabel('Mean Reward (accuracy)')
plt.title(f'GRPO Training Curve — {model_name.split("/")[-1]}')
plt.grid(True, alpha=0.3)
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, 'training_curve.png'), dpi=150)
plt.show()

## Step 7 — Save Checkpoint to Google Drive (optional)

In [ ]:
SAVE_TO_DRIVE = False   # set True to mount Drive and copy checkpoint

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil, yaml
    with open(MODEL_CONFIG) as f:
        cfg_raw = yaml.safe_load(f)
    t = cfg_raw.get('tinker', {})
    checkpoint_dir = t.get('checkpoint_dir', './checkpoints/run/')
    run_name = t.get('wandb_run_name', 'grpo-run')

    drive_dest = f'/content/drive/MyDrive/tinker-rl-checkpoints/{run_name}'
    shutil.copytree(checkpoint_dir, drive_dest, dirs_exist_ok=True)
    print(f'Checkpoint saved to Google Drive: {drive_dest}')
else:
    print('Drive save skipped. Set SAVE_TO_DRIVE=True to enable.')

## Step 8 — Paste Results into Systematic Study Notebook

Copy the `rewards` list printed in Step 6 into `NEW_RUNS` in
`notebooks/systematic_scaling_study.ipynb` under the appropriate key:

```python
NEW_RUNS = {
    # Block A
    'llama_8b_base':     [...],   # paste here
    # Block C
    'qwen_0_6b':         [...],
    'qwen_1_7b':         [...],
    ...
}
```

Then re-run all cells in `systematic_scaling_study.ipynb` to regenerate figures
and statistical tables for inclusion in the report and conference paper.

---
## Quick Reference — All Configs

| Config | Model | Size | LoRA rank | GPU | Block |
|--------|-------|------|-----------|-----|-------|
| `gsm8k_llama_8b_base.yaml` | Llama-3.1-8B (base) | 8B | 32 | A100 40GB | A+B |
| `gsm8k_llama_8b.yaml` | Llama-3.1-8B-Instruct | 8B | 32 | A100 40GB | A |
| `gsm8k_qwen_8b.yaml` | Qwen3-8B (base) | 8B | 32 | A100 40GB | B+C |
| `gsm8k_qwen_0_6b.yaml` | Qwen3-0.6B (base) | 0.6B | 16 | T4 | C |
| `gsm8k_qwen_1_7b.yaml` | Qwen3-1.7B (base) | 1.7B | 16 | T4 | C |
| `gsm8k_qwen_4b.yaml` | Qwen3-4B (base) | 4B | 32 | T4/A100 | C |
| `gsm8k_qwen_14b.yaml` | Qwen3-14B (base) | 14B | 32 | A100 80GB | C |
| `gsm8k_qwen_30b_moe.yaml` | Qwen3-30B-A3B (MoE) | 30B | 32 | A100 80GB | C |

**Cost estimate** (Vast.ai A100 80GB @ ~\$1.50/h):
- 0.6B, 1.7B: free on Colab T4
- 4B, 8B: ~\$3 each on Colab Pro A100
- 14B: ~\$3.50 on Vast.ai A100 80GB
- 30B-MoE: ~\$4 on Vast.ai A100 80GB
- **Total: ~\$20 for full systematic study**